# Проект. Нейросеть для предсказания калорийности блюд

**Автор:** avo_m25_555

**Задача:** по фотографии блюда и списку ингредиентов предсказать калорийность (ккал).

**Целевая метрика:** MAE &lt; 50 на test.

Пайплайн: `scripts/dataset.py`, `scripts/utils.py`, `scripts/config_full.py`. Ноутбук — EDA, `train(cfg)`, инференс.


## Чек-лист ТЗ

| Этап | Содержание |
|------|------------|
| 1 | EDA, merge id→названия, видение решения и метрик |
| 2 | `scripts/dataset.py` + `scripts/utils.py`, `train(cfg)`, seed |
| 3 | Регрессия (head→1, L1Loss, BATCH=4), обучение, чекпоинт |
| 4 | Инференс на test, MAE, топ-5 ошибок с разбором |


## Этап 0. Окружение и путь к `data/`


In [ ]:
# @title 0.1. Зависимости (Colab)
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "timm",
            "albumentations",
            "transformers",
            "torchmetrics",
            "pillow",
            "pandas",
            "tqdm",
            "matplotlib",
        ]
    )
print("IN_COLAB =", IN_COLAB)


In [ ]:
# @title 0.2. Импорты, seed, DATA_ROOT (= папка data/)
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

SEED = 42
IN_COLAB = "google.colab" in sys.modules

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scripts" / "dataset.py").is_file():
    for base in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
        if (base / "scripts" / "dataset.py").is_file() and (base / "avo_m25_555_nutrition.ipynb").is_file():
            NOTEBOOK_DIR = base
            break
sys.path.insert(0, str(NOTEBOOK_DIR))

from scripts.bootstrap import resolve_data_root
from scripts.config_full import get_full_config
from scripts.dataset import load_dish_table, quality_report, rgb_path
from scripts.utils import evaluate, load_model_from_checkpoint, prepare_test_loader, set_seed, train

set_seed(SEED)
NOTEBOOK_DIR, DATA_ROOT = resolve_data_root()
sys.path.insert(0, str(NOTEBOOK_DIR))

ARTIFACTS = Path("/content/nutrition_artifacts") if IN_COLAB else (NOTEBOOK_DIR / "artifacts")
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = str(ARTIFACTS / "best.pt")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DATA_ROOT (data) =", DATA_ROOT)
print("CHECKPOINT_PATH =", CHECKPOINT_PATH)
print("DEVICE =", DEVICE)
print("scripts:", NOTEBOOK_DIR / "scripts")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))


## Этап 1. Исследовательский анализ (EDA)


In [ ]:
# @title 1.1. Загрузка таблиц, merge id → названия, качество данных
dish = load_dish_table(DATA_ROOT)
print("shape:", dish.shape)
print("columns:", dish.columns.tolist())
print("\nsplit:\n", dish["split"].value_counts().to_string())
print("\nпример merge (как в ТЗ):")
raw0 = dish.iloc[0]["ingredients"]
print("  raw ids:", str(raw0)[:80], "...")
print("  text   :", dish.iloc[0]["ingredients_text"][:120])
print("\nпуть фото (ТЗ: data/images/<dish_id>/rgb.png):", dish.iloc[0]["image_path"])
print("rgb_path('data', id) =", rgb_path("data", dish.iloc[0]["dish_id"]))
print("\nhead:")
print(
    dish[["dish_id", "split", "total_calories", "total_mass", "ingredients_text"]]
    .head(5)
    .to_string(index=False)
)
print("\n--- качество ---")
quality_report(dish)


In [ ]:
# @title 1.2. Распределения калорий / массы / числа ингредиентов
dish["n_ingr"] = dish["ingredients_text"].str.split(",").str.len()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col, title in zip(
    axes,
    ["total_calories", "total_mass", "n_ingr"],
    ["Калории (ккал)", "Масса", "Число ингредиентов"],
):
    for split, color in [("train", "C0"), ("test", "C1")]:
        vals = dish.loc[dish["split"] == split, col]
        ax.hist(vals, bins=40, alpha=0.55, label=split, color=color)
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

print(dish.groupby("split")[["total_calories", "total_mass", "n_ingr"]].describe().round(1))


In [ ]:
# @title 1.3. Визуализация примеров (фото + текст + калории)
train_ok = dish[dish["exists"] & (dish["split"] == "train")]
sample = train_ok.sample(n=min(4, len(train_ok)), random_state=SEED)

fig, axes = plt.subplots(1, min(4, len(sample)), figsize=(14, 3.5))
if min(4, len(sample)) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, sample.iterrows()):
    with Image.open(row["image_path"]) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(
        f"{row['total_calories']:.0f} kcal\n{row['ingredients_text'][:40]}…",
        fontsize=8,
    )
    ax.axis("off")
plt.suptitle("Примеры train")
plt.tight_layout()
plt.show()


### Вывод этапа 1 — видение решения

**Как решаем задачу.** Мультимодальная **регрессия**: энкодер изображения (EfficientNet-B0) + энкодер текста ингредиентов (DistilBERT) → проекции → `Linear → 1` (ккал). Вход: `rgb.png` и строка названий продуктов после merge id→`ingr`.

**Данные.** Train/test заданы колонкой `split`. Val — доля train с учётом квантилей калорий. Из train убираем `total_calories <= 0` (аномалия). Test оставляем официальным. Технические `ingr_…` заменены на названия.

**Аугментации.** В выбранном конфиге (после CPU-поиска) слабые/отключённые aug: витринные фото блюд легко «ломаются» сильным ColorJitter/геометрией; важнее стабильный Resize + ImageNet-нормализация. При необходимости — лёгкий HorizontalFlip.

**Метрика.** Основная — **MAE (ккал)** на test; цель продукта MAE &lt; 50. На обучении тот же смысл через `nn.L1Loss`. Дополнительно смотрим val MAE для выбора чекпоинта.


## Этап 2. Пайплайн обучения (`scripts/`)

`dataset.py` — Dataset и loaders. `utils.py` — модель, seed, `train(cfg)`. `config_full.py` — конфиг полного запуска.


In [ ]:
# @title 2.1. Проверка импорта dataset.py
from scripts.dataset import CaloriesDataset, get_eval_transforms, make_loaders, rgb_path

print("rgb_path example:", rgb_path("data", dish.iloc[0]["dish_id"]))
print("Dataset / transforms / loaders: scripts.dataset")


In [ ]:
# @title 2.2. Проверка импорта utils.py / config_full.py
from scripts.config_full import get_full_config
from scripts.utils import Config, MultimodalCalorieModel, build_optimizer, train

print("train(cfg) собирает loaders из cfg.DATA_ROOT")
print("head → 1, L1Loss, AdamW(classifier) — в utils.py")


### Вывод этапа 2

Код обучения воспроизводится вызовом `train(cfg)` из `scripts/utils.py`. Путь к фото: `data/images/<dish_id>/rgb.png`. Оптимизатор включает энкодеры, проекции и `classifier`.


## Этап 3. Обучение модели


In [ ]:
# @title 3.1. train(cfg) по конфиг-файлу
cfg = get_full_config(str(DATA_ROOT), CHECKPOINT_PATH)
print(cfg)
result = train(cfg)


### Вывод этапа 3

Обучение: `BATCH_SIZE=4`, `nn.L1Loss`, `head → 1`. Чекпоинт пишется в `cfg.CHECKPOINT_PATH` (`best.pt` + `best.json`). Test MAE на этапе 4, не во время fit.


## Этап 4. Валидация качества (test MAE + топ-5 ошибок)


In [ ]:
# @title 4.1. Инференс с чекпоинта, test MAE
model, cfg_loaded, ckpt = load_model_from_checkpoint(
    CHECKPOINT_PATH, DEVICE, allowed_dir=ARTIFACTS
)
test_loader, dish_inf, test_df = prepare_test_loader(cfg_loaded)
test_mae = evaluate(model, test_loader, DEVICE)
print(f"TEST MAE = {test_mae:.2f} kcal  |  target < 50: {test_mae < 50}")
print("checkpoint val_mae =", ckpt.get("val_mae"))
print("test size:", len(test_df))


In [ ]:
# @title 4.2. Топ-5 ошибок: фото, ингредиенты, факт vs прогноз
model.eval()
rows = []
with torch.no_grad():
    for batch in test_loader:
        pred = model(
            batch["image"].to(DEVICE),
            batch["input_ids"].to(DEVICE),
            batch["attention_mask"].to(DEVICE),
        ).cpu().numpy()
        y = batch["calories"].numpy()
        for i in range(len(y)):
            rows.append(
                {
                    "dish_id": batch["dish_id"][i],
                    "text": batch["text"][i],
                    "y_true": float(y[i]),
                    "y_pred": float(pred[i]),
                    "abs_err": float(abs(pred[i] - y[i])),
                    "mass": float(batch["mass"][i]),
                }
            )

err_df = pd.DataFrame(rows).sort_values("abs_err", ascending=False)
top5 = err_df.head(5).reset_index(drop=True)
print(top5[["dish_id", "y_true", "y_pred", "abs_err", "mass"]].to_string(index=False))

id2path = {str(k): v for k, v in dish.set_index("dish_id")["image_path"].to_dict().items()}
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, (_, r) in zip(axes, top5.iterrows()):
    path = id2path[str(r["dish_id"])]
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(
        f"true {r['y_true']:.0f} / pred {r['y_pred']:.0f}\nerr {r['abs_err']:.0f}\n{r['text'][:36]}…",
        fontsize=7,
    )
    ax.axis("off")
plt.suptitle("Топ-5 ошибок на test")
plt.tight_layout()
plt.show()

for i, r in top5.iterrows():
    print(f"\n#{i+1} {r['dish_id']}")
    print(f"  ingredients: {r['text']}")
    print(
        f"  true={r['y_true']:.1f}  pred={r['y_pred']:.1f}  "
        f"|err|={r['abs_err']:.1f}  mass={r['mass']:.1f}"
    )


### Вывод этапа 4 — разбор ошибок

После прогона ячейки 4.2 разбор строится по конкретным `dish_id` (фото, ингредиенты, факт vs прогноз). Типовые гипотезы, если |err| большой:

1. **Нетипичная масса порции** при похожем составе — калории зависят от граммовки сильнее, чем от вида тарелки.
2. **Редкие / смешанные ингредиенты** — мало примеров в train.
3. **Визуально похожие блюда** с разной калорийностью (соусы, масло не видны на rgb).
4. **Качество кадра** (ракурс, обрезка, свет).

Если test MAE ≥ 50, эти примеры — гипотезы для доработки (признак `total_mass`, балансировка по калориям).
